600 images  (using 70-30 split, with 70 % training further split into 75% training and 25% validation)
       (180 testing images; 420 training images- 315 training 105 validation images)
Comdoenca – with the disease
Endoenca- without the disease

Image split
testing:
90 com, 90 sem
training:
315
157 com 158 sem
validation: 105
53 com 52 sem

Using transfer learning: VGG19, EfficientNetB3, ResNet50, DenseNet121

Fixed: drop-out (0.6, 0.4, 0.3), dense nodes (256, 128, 64), no of epochs:50, input size (256x256); batch size 32
 
Apply pre-processing techniques: normalization, image shuffling, augmentation, train-test split
Should have augmentation on the training images only

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import ops
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, GlobalAveragePooling2D, MaxPool2D, BatchNormalization
from keras.applications.resnet50 import ResNet50
from keras.applications.vgg19 import VGG19
from keras.applications.efficientnet import EfficientNetB3
from keras.applications.densenet import DenseNet121
from keras import backend as K

In [2]:
for dirname, _, filenames in os.walk('Dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


Dataset\test\comdoença\cd (1).jpg
Dataset\test\comdoença\cd (10).jpg
Dataset\test\comdoença\cd (11).jpg
Dataset\test\comdoença\cd (12).jpg
Dataset\test\comdoença\cd (13).jpg
Dataset\test\comdoença\cd (14).jpg
Dataset\test\comdoença\cd (15).jpg
Dataset\test\comdoença\cd (16).jpg
Dataset\test\comdoença\cd (17).jpg
Dataset\test\comdoença\cd (18).jpg
Dataset\test\comdoença\cd (19).jpg
Dataset\test\comdoença\cd (2).jpg
Dataset\test\comdoença\cd (20).jpg
Dataset\test\comdoença\cd (21).jpg
Dataset\test\comdoença\cd (22).jpg
Dataset\test\comdoença\cd (23).jpg
Dataset\test\comdoença\cd (24).jpg
Dataset\test\comdoença\cd (25).jpg
Dataset\test\comdoença\cd (26).jpg
Dataset\test\comdoença\cd (27).jpg
Dataset\test\comdoença\cd (28).jpg
Dataset\test\comdoença\cd (29).jpg
Dataset\test\comdoença\cd (3).jpg
Dataset\test\comdoença\cd (30).jpg
Dataset\test\comdoença\cd (31).jpg
Dataset\test\comdoença\cd (32).jpg
Dataset\test\comdoença\cd (33).jpg
Dataset\test\comdoença\cd (34).jpg
Dataset\test\comdoença\

In [3]:
image_generatorFullAugment = ImageDataGenerator(
    rotation_range = 30,
    width_shift_range = 0.15,
    shear_range = 0.3,
    samplewise_center=True,
    horizontal_flip = True,
    samplewise_std_normalization = True
)

image_generatorNoAugment = ImageDataGenerator(
    samplewise_std_normalization = True
)

c:\Users\gab\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\keras\src\legacy\preprocessing\image.py:1070: UserWarning: This ImageDataGenerator specifies `samplewise_std_normalization`, which overrides setting of `samplewise_center`.
  warnings.warn(


In [4]:
train = image_generatorFullAugment.flow_from_directory(
    "Dataset/train",
    batch_size = 32,
    class_mode = 'binary',
    target_size=(256, 256)
)

validation = image_generatorFullAugment.flow_from_directory(
    "Dataset/validation",
    batch_size = 1,
    class_mode = 'binary',
    target_size=(256, 256)
)

test = image_generatorFullAugment.flow_from_directory(
    "Dataset/test",
    batch_size = 1,
    class_mode = 'binary',
    target_size=(256, 256)
)

Found 315 images belonging to 2 classes.
Found 105 images belonging to 2 classes.
Found 180 images belonging to 2 classes.


Main Experiments: to run at LR=0.0001, 0.00001, 0.000001
Adam (Adaptive Moment Estimation optimizer)
Adagrad (Adaptive Gradient optimizer) 
Adamax (Maximum adaptive moment estimation optimizer)
AdaDelta
SGD (Stochastic Gradient Descent)
RMSProp (Root Mean Square Propagation optimizer)
	                  
Main Metric: Accuracy and AUC; but need to get all metrics (accuracy, recall, precision, specificity, F1-score and AUC)
Binary classification: With Disease (COmdoenca) vs Without Disease (Endoenca)   
 TOTAL MODELS: 3 learning rates x 6 optimizers x 4 pretrained models = 72 simulations

In [5]:
def custom_precision(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def custom_recall(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def custom_f1(y_true, y_pred):
    precision = custom_precision(y_true, y_pred)
    recall = custom_recall(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))


In [ ]:
def set_model(model_name, lr, optimizer_name):
    if model_name == "resnet":
        base_model = ResNet50(include_top=False, weights='imagenet')
    elif model_name == "densenet":
        base_model = DenseNet121(include_top=False, weights='imagenet')
    if model_name == "efficientnet":
        base_model = EfficientNetB3(include_top=False, weights='imagenet')
    elif model_name == "vgg":
        base_model = VGG19(include_top=False, weights='imagenet')

    base_model.trainable = False
        #resnet_base_model.summary()

    model = tf.keras.Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation = "relu"),
        BatchNormalization(),
        Dropout(0.6),
        Dense(128, activation = "relu"),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation = "relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(1, activation="sigmoid")
    ])

    if optimizer_name == "adam":
        opt = tf.keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name == "adagrad":
        opt = tf.keras.optimizers.Adagrad(learning_rate=lr)
    elif optimizer_name == "adamax":
        opt = tf.keras.optimizers.Adamax(learning_rate=lr)
    elif optimizer_name == "adadelta":
        opt = tf.keras.optimizers.Adadelta(learning_rate=lr)
    elif optimizer_name == "sgd":
        opt = tf.keras.optimizers.SGD(learning_rate=lr)
    elif optimizer_name == "rmsp":
        opt = tf.keras.optimizers.RMSprop(learning_rate=lr)


    metrics = [
        'accuracy',
        tf.keras.metrics.Precision(name = 'precision'),
        tf.keras.metrics.Recall(name = 'recall'),
        tf.keras.metrics.AUC(name = 'auc'), #main
        #tf.keras.metrics.Accuracy(name = 'accuracy'), #main
        tf.keras.metrics.SpecificityAtSensitivity(sensitivity =0.5, num_thresholds = 200, name = 'specificity')#,
        #custom_f1
    ]

    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=metrics)
    run_model = model.fit(train, epochs = 50, validation_data= validation)
    pred_run_model = run_model.predict(test)
    print(confusion_matrix(test.classes, pred_run_model > 0.5))
    pd.DataFrame(classification_report(test.classes, pred_run_model > 0.5, output_dict = True))

In [ ]:
#resnet
set_model("resnet", 0.0001, "adam")
set_model("resnet", 0.00001, "adam")
set_model("resnet", 0.000001, "adam")

set_model("resnet", 0.0001, "adagrad")
set_model("resnet", 0.00001, "adagrad")
set_model("resnet", 0.000001, "adagrad")

set_model("resnet", 0.0001, "adamax")
set_model("resnet", 0.00001, "adamax")
set_model("resnet", 0.000001, "adamax")

set_model("resnet", 0.0001, "adadelta")
set_model("resnet", 0.00001, "adadelta")
set_model("resnet", 0.000001, "adadelta")

set_model("resnet", 0.0001, "sgd")
set_model("resnet", 0.00001, "sgd")
set_model("resnet", 0.000001, "sgd")

set_model("resnet", 0.0001, "rmsp")
set_model("resnet", 0.00001, "rmsp")
set_model("resnet", 0.000001, "rmsp")

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 0.5270 - accuracy_1: 0.0000e+00 - auc: 0.5349 - loss: 0.9165 - precision: 0.5269 - recall: 0.5570 - specificity: 0.5478 - val_accuracy: 0.6381 - val_accuracy_1: 0.0000e+00 - val_auc: 0.6245 - val_loss: 0.6777 - val_precision: 0.6591 - val_recall: 0.5577 - val_specificity: 0.7547
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 0.5492 - accuracy_1: 0.0000e+00 - auc: 0.5804 - loss: 0.9064 - precision: 0.5471 - recall: 0.5886 - specificity: 0.6242 - val_accuracy: 0.5714 - val_accuracy_1: 0.0000e+00 - val_auc: 0.6560 - val_loss: 0.6738 - val_precision: 0.5467 - val_recall: 0.7885 - val_specificity: 0.7736
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 42s 4s/step - accuracy: 0.5556 - accuracy_1: 0.0000e+00 - auc: 0.5805 - loss: 0.8779 - precision: 0.5529 - recall: 0.5949 - specificity: 0.5796 - val_accuracy: 0.5143 - val_accuracy_1: 0.0000e+00 - val_auc: 0.5898 - val_loss: 0.6977 - val_precision: 0.5057 - val_recall: 0.8462 

In [ ]:
#densenet
set_model("densenet", 0.0001, "adam")
set_model("densenet", 0.00001, "adam")
set_model("densenet", 0.000001, "adam")

set_model("densenet", 0.0001, "adagrad")
set_model("densenet", 0.00001, "adagrad")
set_model("densenet", 0.000001, "adagrad")

set_model("densenet", 0.0001, "adamax")
set_model("densenet", 0.00001, "adamax")
set_model("densenet", 0.000001, "adamax")

set_model("densenet", 0.0001, "adadelta")
set_model("densenet", 0.00001, "adadelta")
set_model("densenet", 0.000001, "adadelta")

set_model("densenet", 0.0001, "sgd")
set_model("densenet", 0.00001, "sgd")
set_model("densenet", 0.000001, "sgd")

set_model("densenet", 0.0001, "rmsp")
set_model("densenet", 0.00001, "rmsp")
set_model("densenet", 0.000001, "rmsp")

180/180 ━━━━━━━━━━━━━━━━━━━━ 25s 122ms/step
[[88  2]
 [86  4]]


,0,1,accuracy,macro avg,weighted avg
precision,0.505747,0.666667,0.511111,0.586207,0.586207
recall,0.977778,0.044444,0.511111,0.511111,0.511111
f1-score,0.666667,0.083333,0.511111,0.375000,0.375000
support,90.000000,90.000000,0.511111,180.000000,180.000000
